# Mushroom Manifest Builder

This notebook scans a folder (e.g., `TexturePack/mushroom_pack/`) for your generated PNGs
using the **new filename format**:

`color-stem-cap-value-r0/r1-rp(position)-pt(pattern).png`

Examples:
- `black-03-0.201-+0-r0-rpnone-pt0.png`
- `black-06-1.184-+2-r1-rptop-pt0.png`
- `black-06-1.154-+2-r1-rpmiddle-pt1.png`
- `black-09-1.230-+2-r1-rpbottom-pt2.png`

Then it writes a `manifest.json` into that folder. Your web app can fetch this manifest at runtime.

## What it does
- Recursively scans for files ending in `.png`.
- Validates filenames against the new naming convention.
- Builds an array of file paths (either basenames or paths relative to the base folder).
- Saves `manifest.json` in the base folder.
- Prints helpful stats.

## How to use
1. **Edit** the `BASE_DIR` below to the absolute path of your folder on *your* machine.
2. Run each cell top-to-bottom.
3. Confirm the printed stats look right.
4. Check that `manifest.json` exists under `BASE_DIR`.

> Tip: If your JS expects basenames only, set `WRITE_BASENAMES = True`. If you prefer
> manifest entries to include the folder path, set it to `False` to write paths **relative**
> to `BASE_DIR`.


In [2]:
# === Use your exact local folder for scanning (only used to FIND files) ===
BASE_DIR = r"/Users/jianleguo/Documents/GitHub/vmps_mario/TexturePack/mushroom_pack_second_version"

# Write only filenames like "black-06-1.184-+2-r1-rptop-pt0.png" (no paths in manifest)
WRITE_BASENAMES = True

# Name of the output manifest placed INSIDE BASE_DIR
MANIFEST_NAME = "manifest.json"

import os
assert os.path.isdir(BASE_DIR), f"BASE_DIR does not exist: {BASE_DIR}"
print("✓ Using BASE_DIR:", BASE_DIR)
print("Manifest will contain basenames only:", WRITE_BASENAMES)


✓ Using BASE_DIR: /Users/jianleguo/Documents/GitHub/vmps_mario/TexturePack/mushroom_pack_second_version
Manifest will contain basenames only: True


In [3]:

import os, re, json
from collections import Counter

# New filename format:
# color-stem-cap-value-r0/r1-rp(position)-pt(pattern).png
pattern = re.compile(
    r"^([a-z]+)-"                 # color
    r"(\d+)-"                    # stem width
    r"(\d+\.\d+)-"               # cap roundness
    r"([+-]?\d+)-"               # assigned value
    r"(r[01])-"
    r"(rp(?:none|top|middle|bottom))-"
    r"(pt[012])\.png$",
    re.I
)

def basename(p: str) -> str:
    return os.path.basename(p)

def is_valid_name(name: str) -> bool:
    return pattern.match(name) is not None

def scan_pngs(base_dir: str, write_basenames: bool = True):
    found = []
    invalid = []
    for root, _, files in os.walk(base_dir):
        for f in files:
            if not f.lower().endswith(".png"):
                continue
            if is_valid_name(f):
                if write_basenames:
                    found.append(f)
                else:
                    rel = os.path.relpath(os.path.join(root, f), base_dir)
                    found.append(rel.replace("\\", "/"))
            else:
                invalid.append(f)
    return found, invalid

def parse_name(name: str):
    m = pattern.match(name)
    if not m:
        return None
    return {
        "color": m.group(1).lower(),
        "stem_width": int(m.group(2)),
        "cap_roundness": float(m.group(3)),
        "assigned_value": int(m.group(4)),
        "have_ring": m.group(5).lower() == "r1",
        "ring_position": m.group(6)[2:].lower(),   # drop "rp"
        "pattern_type": int(m.group(7)[2:]),       # drop "pt"
    }

def color_from_name(name: str) -> str:
    parsed = parse_name(name)
    return parsed["color"] if parsed else "invalid"

def ring_position_from_name(name: str) -> str:
    parsed = parse_name(name)
    return parsed["ring_position"] if parsed else "invalid"

def pattern_type_from_name(name: str) -> int:
    parsed = parse_name(name)
    return parsed["pattern_type"] if parsed else -1


In [4]:

# --- Scan ---
assert os.path.isdir(BASE_DIR), f"BASE_DIR does not exist: {BASE_DIR}"
found, invalid = scan_pngs(BASE_DIR, write_basenames=WRITE_BASENAMES)

print(f"Scanned folder: {BASE_DIR}")
print(f"Valid PNGs found: {len(found)}")
print(f"Invalid (skipped): {len(invalid)}")
if invalid:
    print("Some invalid samples:", invalid[:10])

# --- Stats ---
by_color = Counter(color_from_name(os.path.basename(x)) for x in found)
by_ring_position = Counter(ring_position_from_name(os.path.basename(x)) for x in found)
by_pattern_type = Counter(pattern_type_from_name(os.path.basename(x)) for x in found)

print("\nCounts by color:", dict(by_color))
print("Counts by ring position:", dict(by_ring_position))
print("Counts by pattern type:", dict(by_pattern_type))

print("\nSample entries:", found[:10])

# --- Save manifest ---
manifest_path = os.path.join(BASE_DIR, MANIFEST_NAME)
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(found, f, ensure_ascii=False, indent=2)

print(f'\nWrote manifest with {len(found)} entries -> {manifest_path}')


Scanned folder: /Users/jianleguo/Documents/GitHub/vmps_mario/TexturePack/mushroom_pack_second_version
Valid PNGs found: 3200
Invalid (skipped): 1
Some invalid samples: ['rainbow_mushroom.png']

Counts by color: {'cyan': 400, 'black': 400, 'blue': 400, 'red': 400, 'yellow': 400, 'green': 400, 'white': 400, 'magenta': 400}
Counts by ring position: {'none': 1591, 'bottom': 530, 'top': 516, 'middle': 563}
Counts by pattern type: {1: 806, 2: 834, 0: 1560}

Sample entries: ['cyan-14-1.402-+1-r0-rpnone-pt1.png', 'black-14-1.049-+6-r0-rpnone-pt2.png', 'blue-04-0.202-+1-r1-rpbottom-pt1.png', 'black-14-0.203-+5-r0-rpnone-pt2.png', 'red-14-0.759-+9-r1-rptop-pt0.png', 'yellow-07-0.645-+2-r1-rpbottom-pt1.png', 'blue-14-1.506--1-r1-rpmiddle-pt0.png', 'green-13-1.091-+3-r0-rpnone-pt1.png', 'green-03-0.530-+2-r1-rpbottom-pt2.png', 'white-03-1.861--1-r0-rpnone-pt1.png']

Wrote manifest with 3200 entries -> /Users/jianleguo/Documents/GitHub/vmps_mario/TexturePack/mushroom_pack_second_version/manifest.js

In [5]:

# Re-open and validate the just-written manifest
manifest_path = os.path.join(BASE_DIR, MANIFEST_NAME)
with open(manifest_path, "r", encoding="utf-8") as f:
    data = json.load(f)

assert isinstance(data, list), "Manifest JSON must be an array."
bad = [x for x in data if not is_valid_name(os.path.basename(x))]
print(f"Re-loaded {len(data)} entries from manifest.")
print("Invalid names in manifest (should be 0):", len(bad))
if bad:
    print(bad[:10])


Re-loaded 3200 entries from manifest.
Invalid names in manifest (should be 0): 0
